In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
data = pd.read_csv('krisha_clean.csv')

X = data.drop('price', axis=1)
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20]
}

rf_model = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=params,
    cv=5,
    scoring='r2'
)

rf_model.fit(X_train, y_train)

print(f"Лучшие параметры: {rf_model.best_params_}")

Лучшие параметры: {'max_depth': 20, 'n_estimators': 300}


In [8]:
y_pred_lr = lr_model.predict(X_test_scaled)

y_pred_rf = rf_model.best_estimator_.predict(X_test)

In [9]:
lr_cv = cross_val_score(
    LinearRegression(), 
    X_train_scaled, 
    y_train, 
    cv=5, 
    scoring='r2')

rf_cv = cross_val_score(
    rf_model.best_estimator_, 
    X_train, 
    y_train, 
    cv=5, 
    scoring='r2')

print(f"Linear Regression CV R²: {lr_cv.mean():.3f} (+/- {lr_cv.std():.3f})")
print(f"Random Forest CV R²:     {rf_cv.mean():.3f} (+/- {rf_cv.std():.3f})")

Linear Regression CV R²: 0.731 (+/- 0.022)
Random Forest CV R²:     0.804 (+/- 0.009)


In [10]:
mse_lr = mean_absolute_error(y_test, y_pred_lr)
mse_rf = mean_absolute_error(y_test, y_pred_rf)

rmse_lr = np.sqrt(mse_lr)
rmse_rf = np.sqrt(mse_rf)

r2_lr = r2_score(y_test, y_pred_lr)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Linear Regression MAE: {mse_lr:.2f}, RMSE: {rmse_lr:.2f}, R²: {r2_lr:.3f}")
print(f"Random Forest MAE:     {mse_rf:.2f}, RMSE: {rmse_rf:.2f}, R²: {r2_rf:.3f}")

Linear Regression MAE: 14436488.53, RMSE: 3799.54, R²: 0.697
Random Forest MAE:     10590039.24, RMSE: 3254.23, R²: 0.801
